# Installation

In [24]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install openai

  Using cached openai-1.68.2-py3-none-any.whl.metadata (25 kB)
Using cached openai-1.68.2-py3-none-any.whl (606 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain_openai

  Using cached langchain_openai-0.3.10-py3-none-any.whl.metadata (2.3 kB)
Using cached langchain_openai-0.3.10-py3-none-any.whl (61 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install chromadb


  Using cached chromadb-0.6.3-py3-none-any.whl.metadata (6.8 kB)
  Using cached chroma_hnswlib-0.7.6-cp311-cp311-macosx_11_0_arm64.whl.metadata (252 bytes)
  Using cached numpy-2.2.4-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pydantic-1.10.21-cp311-cp311-macosx_11_0_arm64.whl.metadata (153 kB)
Using cached chromadb-0.6.3-py3-none-any.whl (611 kB)
Using cached chroma_hnswlib-0.7.6-cp311-cp311-macosx_11_0_arm64.whl (185 kB)
Using cached numpy-2.2.4-cp311-cp311-macosx_14_0_arm64.whl (5.4 MB)
Using cached pydantic-1.10.21-cp311-cp311-macosx_11_0_arm64.whl (2.6 MB)
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.10.6
    Uninstalling pydantic-2.10.6:
      Successfully uninstalled pydantic-2.10.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seaborn 0.13.2 requires pandas>=1.2, which is not installed.
langchain 0

In [4]:
%pip install chromadb openai 

Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install langchain_community

  Using cached pydantic-2.10.6-py3-none-any.whl.metadata (30 kB)
Using cached pydantic-2.10.6-py3-none-any.whl (431 kB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install unstructured

  Using cached unstructured-0.17.2-py3-none-any.whl.metadata (24 kB)
Using cached unstructured-0.17.2-py3-none-any.whl (1.8 MB)
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install python-pptx

  Using cached python_pptx-1.0.2-py3-none-any.whl.metadata (2.5 kB)
Using cached python_pptx-1.0.2-py3-none-any.whl (472 kB)
Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install langchain_chroma

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install pandas

  Using cached pandas-2.2.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (89 kB)
Using cached pandas-2.2.3-cp311-cp311-macosx_11_0_arm64.whl (11.3 MB)
Note: you may need to restart the kernel to use updated packages.


# Importing Libraries

In [43]:
import openai
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredPowerPointLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import unstructured
import pptx
import chromadb
from chromadb.utils import embedding_functions
from langchain_chroma import Chroma
from typing import List, Tuple

# API Key

In [44]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Vectorize Model and Importing the Files into ChromaDB

In [49]:
class Vectorize:
    def __init__(self, folder_location="./testfiles", openai_api_key="SOME DEFAULT KEY", collection_name="Sample"):
        self.folder_location = folder_location
        self.openai_api_key = openai_api_key
        
        documents: List[Tuple[str, str]] = []

        # Processing texts from a folder of pptx and txt files

        for filename in os.listdir(folder_location):
            filepath = os.path.join(folder_location, filename)
            if filename.endswith(".pptx"):
                try:
                    loader = UnstructuredPowerPointLoader(filepath)
                    loaded_docs = loader.load()
                    for doc in loaded_docs:
                        documents.append((doc.page_content, {"source": filename}))
                except Exception as e:
                    print(f"Error loading {filename}: {e}")

            elif filename.endswith(".txt"):
                try:
                    loader = TextLoader(filepath)
                    loaded_docs = loader.load()
                    for doc in loaded_docs:
                        documents.append((doc.page_content, {"source": filename}))
                except Exception as e:
                    print(f"Error loading {filename}: {e}")

            else:
                print("Only .txt and .pptx files are allowed!")

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        texts = text_splitter.create_documents([doc[0] for doc in documents], metadatas=[doc[1] for doc in documents])

        #print("Initializing OpenAI embeddings...")
        self.embeddings = embedding_functions.OpenAIEmbeddingFunction(api_key=openai_api_key)

        client = chromadb.PersistentClient(path="/tmp/chromadb")
        collection = client.get_or_create_collection(name=collection_name, embedding_function=self.embeddings)

        collection.add(
            documents=[doc.page_content for doc in texts],
            metadatas=[doc.metadata for doc in texts],
            ids=[f"doc_{i}" for i in range(len(texts))],
        )

        self.db = collection

        print(f"Successfully embedded and stored {len(texts)} chunks in ChromaDB.")

    def query_text(self, input_text):
        results = self.db.query(
            query_texts = [input_text],
            n_results=3)
        if results and 'documents' in results and 'metadatas' in results:
          for i in range(len(results['documents'][0])):
              print(f"* {results['documents'][0][i]} [{results['metadatas'][0][i]}]")
              print("-" * 20)
    
    def get_one_embedding(self, input_id):
        results = self.db.get(ids=[input_id], include=["embeddings"])
        return results['embeddings'][0]
        
    
    def query_embedding(self, input_embedding):
        results = self.db.query(
            query_embeddings = [input_embedding],
            include=['embeddings', 'metadatas', 'documents', 'distances'],
            n_results=3)
        if results and 'embeddings' in results and 'documents' in results and 'metadatas' in results and 'distances' in results :
            for i in range(len(results['documents'][0])):
                print(f"Source File: {results['metadatas'][0][i]}")
                print(f"Distance: {results['distances'][0][i]}") # print the distances
                print(f"Embedding: {results['embeddings'][0][i]}")
                print("-" * 20)
        else:
            print("No results found or unexpected result structure.")




In [50]:
folder_location = input("Place the name of the folder: ") # folder where there are 10 pptx and txt files
cmd = Vectorize(folder_location=folder_location, openai_api_key=OPENAI_API_KEY)

Insert of existing embedding ID: doc_0
Insert of existing embedding ID: doc_1
Insert of existing embedding ID: doc_2
Insert of existing embedding ID: doc_3
Insert of existing embedding ID: doc_4
Insert of existing embedding ID: doc_5
Insert of existing embedding ID: doc_6
Insert of existing embedding ID: doc_7
Insert of existing embedding ID: doc_8
Insert of existing embedding ID: doc_9
Add of existing embedding ID: doc_0
Add of existing embedding ID: doc_1
Add of existing embedding ID: doc_2
Add of existing embedding ID: doc_3
Add of existing embedding ID: doc_4
Add of existing embedding ID: doc_5
Add of existing embedding ID: doc_6
Add of existing embedding ID: doc_7
Add of existing embedding ID: doc_8
Add of existing embedding ID: doc_9


Successfully embedded and stored 10 chunks in ChromaDB.


# Query Functions

In [51]:
# Query for text (Testing if the query works)
input_text = input()
cmd.query_text(input_text=input_text)

* If several anonymous parties are referenced, they may simply be labelled John Doe #1, John Doe #2, etc. (the U.S. Operation Delego cited 21 (numbered) "John Doe"s) or labelled with other variants of Doe / Roe / Poe / etc. Other early alternatives such as John Stiles and Richard Miles are now rarely used, and Mary Major has been used in some American federal cases. [{'source': 'test5.txt'}]
--------------------
* The names "John Doe" for males, "Jane Doe" or "Jane Roe" for females, or "Jonnie Doe" and "Janie Doe" for children, or just "Doe" non-gender-specifically are used as placeholder names for a party whose true identity is unknown or must be withheld in a legal action, case, or discussion. [{'source': 'test.txt'}]
--------------------
* John Doe is sometimes used to refer to a typical male in other contexts as well, in a similar manner to John Q. Public, known in Great Britain as Joe Public, John Smith or Joe Bloggs. For example, the first name listed on a form is often John Doe,

In [52]:
# Provide a variable to place a sample embedding due to the high dimensionality of the embedding
id = int(input())
sample_embedding = cmd.get_one_embedding(input_id=f"doc_{id}")

In [53]:
# Query for embedding
cmd.query_embedding(
    input_embedding=sample_embedding
)

Source File: {'source': 'test5.txt'}
Distance: 0.0
Embedding: [-0.0295921  -0.00950132 -0.02366041 ... -0.01999789 -0.00595823
 -0.01082832]
--------------------
Source File: {'source': 'test.txt'}
Distance: 0.160840300174645
Embedding: [-0.02987485 -0.01375579 -0.01554109 ... -0.02897578  0.00295891
 -0.00056914]
--------------------
Source File: {'source': 'test4.txt'}
Distance: 0.2109428370987997
Embedding: [-0.02203982 -0.0169686  -0.01305227 ... -0.00554544 -0.02283986
 -0.00265336]
--------------------
